<a href="https://colab.research.google.com/github/zeynepdanis/kriptografi-colab/blob/main/Kriptografi_AES128_ECDSA_P256.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔐 Kriptografi Algoritmaları: AES-128 & ECDSA P-256




## 📋 İçindekiler

### Bölüm 1 — AES-128 Simetrik Şifreleme
1. [Temel Tablolar: S-Box, InvS-Box, RCON](#1)
2. [GF(2⁸) Aritmetiği: xtime & gmul](#2)
3. [Yardımcı Fonksiyonlar](#3)
4. [Anahtar Genişletme (Key Expansion)](#4)
5. [Şifreleme Dönüşümleri](#5)
6. [Şifre Çözme (Ters Dönüşümler)](#6)
7. [PKCS#7 Dolgu ve Kullanıcı Arayüzü](#7)
8. [Demo: Şifrele & Çöz](#8)
9. [Uygulama Örneği: Güvenli Mesaj Şifreleme (CBC)](#app-aes)

### Bölüm 2 — ECDSA P-256 Dijital İmza
9. [Eğri Parametreleri (P-256)](#9)
10. [Modüler Ters — mod_inv](#10)
11. [Eliptik Eğri Nokta Aritmetiği](#11)
12. [Yardımcı Fonksiyonlar (SHA-256, Rastgele)](#12)
13. [Anahtar Üretimi](#13)
14. [İmzalama](#14)
15. [Doğrulama](#15)
16. [Tahrif Testi](#16)
17. [Demo: Uçtan Uca ECDSA](#17)
18. [Uygulama Örneği: Blok Zinciri İşlem İmzalama](#app-ecdsa)


---
## 🔷 BÖLÜM 1: AES-128 — Simetrik Blok Şifreleme

> **AES (Advanced Encryption Standard)**, NIST tarafından 2001 yılında standart olarak kabul edilmiş simetrik blok şifrelemedir.
> 128-bit sürümü 16 baytlık bloklar üzerinde **10 tur** çalışır.
> Bu implementasyonda hiçbir kriptografi kütüphanesi kullanılmamıştır — her şey sıfırdan yazılmıştır.

| Parametre | Değer |
|-----------|-------|
| Blok Boyutu | 128 bit (16 bayt) |
| Anahtar Boyutu | 128 bit (16 bayt) |
| Tur Sayısı | 10 (+ 1 ön tur) |
| Durum Matrisi | 4×4 bayt |
| Güvenlik | 128 bit |


<a id='1'></a>
###  Temel Tablolar: S-Box, InvS-Box ve RCON

**S-Box**, AES'in doğrusal olmayan tek bileşenidir — her baytı farklı bir değere eşler ve kriptografik güç sağlar.
**InvS-Box**, S-Box'ın tersidir; şifre çözmede kullanılır.
**RCON** (Round Constant), anahtar genişletme sırasında XOR'lanan GF(2⁸) tabanlı sabitlerdir.


In [ ]:
# ─────────────────────────────────────────────────────────
#  S-Box: AES'in temel ikame tablosu (256 eleman)
#  Doğrusal olmayan dönüşüm → kriptografik güç sağlar
# ─────────────────────────────────────────────────────────
SBOX = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16,
]

# InvS-Box: S-Box'ın matematiksel tersi (şifre çözmede kullanılır)
INV_SBOX = [0] * 256
for i, v in enumerate(SBOX):
    INV_SBOX[v] = i

# RCON: Anahtar genişletmede kullanılan tur sabitleri
# GF(2^8) alanında 2'nin kuvvetleri: 1, 2, 4, 8, ...
RCON = [0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1b, 0x36]

print(f"S-Box kontrolü  — SBOX[0x00] = {SBOX[0x00]:02X}  (beklenen: 63)")
print(f"InvS-Box kontrolü — INV_SBOX[0x63] = {INV_SBOX[0x63]:02X}  (beklenen: 00)")
print(f"RCON uzunluğu: {len(RCON)} eleman (10 tur için yeterli)")


S-Box kontrolü  — SBOX[0x00] = 63  (beklenen: 63)
InvS-Box kontrolü — INV_SBOX[0x63] = 00  (beklenen: 00)
RCON uzunluğu: 10 eleman (10 tur için yeterli)


<a id='2'></a>
### GF(2⁸) Aritmetiği: `xtime` ve `gmul`

MixColumns, **GF(2⁸)** (Galois Alanı) üzerinde çarpma gerektirir. Bu alan, indirgeyici polinom
`x⁸ + x⁴ + x³ + x + 1 = 0x11b` ile tanımlanır.

| Fonksiyon | Açıklama |
|-----------|----------|
| `xtime(b)` | GF(2⁸)'de `b × 2`: bir bit sola kaydır, taşma varsa `0x1b` ile XOR |
| `gmul(a, b)` | GF(2⁸)'de genel çarpma: Russian Peasant yöntemi |


In [ ]:
def xtime(b):
    """
    GF(2^8) alanında 2 ile çarpma.
    Bit 7 = 1 ise taşma olur → 0x1b ile XOR (indirgeme polinomu)
    """
    return ((b << 1) ^ 0x1b) & 0xff if b & 0x80 else (b << 1) & 0xff


def gmul(a, b):
    """
    GF(2^8) alanında genel çarpma (Russian Peasant / ikili çarpma).
    Her iterasyonda:
      - b'nin en sağ biti 1 ise mevcut a'yı sonuca ekle (XOR)
      - a'yı xtime ile ilerlet (2 ile çarp)
      - b'yi bir sağa kaydır
    """
    result = 0
    for _ in range(8):
        if b & 1:
            result ^= a
        a = xtime(a)
        b >>= 1
    return result


# Doğrulama
print(f"xtime(0x57) = {xtime(0x57):02X}  (beklenen: AE)")
print(f"gmul(0x57, 0x13) = {gmul(0x57, 0x13):02X}  (beklenen: FE — NIST test vektörü)")


xtime(0x57) = AE  (beklenen: AE)
gmul(0x57, 0x13) = FE  (beklenen: FE — NIST test vektörü)


<a id='3'></a>
### Yardımcı Fonksiyonlar

AES, 16 baytlık veriyi sütun-öncelikli (**column-major**) düzende 4×4 matrise yerleştirir.
Bu fonksiyonlar veri ve matris arasındaki dönüşümü sağlar.


In [ ]:
def matrix_to_hex(state):
    """4x4 durum matrisini okunabilir hex string'e çevirir."""
    result = ""
    for row in range(4):
        for col in range(4):
            result += f"{state[row][col]:02X} "
        result += "\n"
    return result.strip()


def bytes_to_state(data: bytes) -> list:
    """
    16 baytı 4×4 matrise dönüştürür (sütun-öncelikli yerleşim).
    Bayt i → satır = i%4, sütun = i//4
    """
    state = [[0]*4 for _ in range(4)]
    for i in range(16):
        state[i % 4][i // 4] = data[i]
    return state


def state_to_bytes(state: list) -> bytes:
    """4×4 matrisi tekrar 16 bayta dönüştürür (sütun-öncelikli)."""
    result = []
    for col in range(4):
        for row in range(4):
            result.append(state[row][col])
    return bytes(result)


def print_step(baslik, state, aciklama=""):
    """Tur içi görselleştirme: adım başlığı + 4×4 matris."""
    print(f"\n  ┌─ {baslik} {'─'*(40-len(baslik))}")
    if aciklama:
        print(f"  │  {aciklama}")
        print(f"  │")
    for row in range(4):
        satir = "  │   "
        for col in range(4):
            satir += f"{state[row][col]:02X}  "
        print(satir)
    print(f"  └{'─'*45}")


<a id='4'></a>
###  Anahtar Genişletme (Key Expansion)

128-bit ana anahtardan **11 tur anahtarı** (44 kelime × 4 bayt) üretilir.
Her yeni kelime, önceki 4. kelimenin XOR'u + üç özel dönüşümdür:

1. **RotWord** — kelimenin baytlarını sola döndür
2. **SubWord** — her bayta S-Box uygula  
3. **Rcon XOR** — ilgili tur sabitiyle XOR


In [ ]:
def key_expansion(key: bytes) -> list:
    """
    128-bit anahtardan 11 tur anahtarı üretir.
    Döndürülen w listesi: 44 kelime (her kelime 4 bayt).
    get_round_key(w, r) ile r. tur anahtarı alınır.
    """
    w = []
    # İlk 4 kelime doğrudan anahtardan alınır
    for i in range(4):
        w.append(list(key[i*4 : i*4+4]))

    for i in range(4, 44):
        temp = w[i-1][:]
        if i % 4 == 0:
            temp = temp[1:] + temp[:1]          # RotWord: sola döndür
            temp = [SBOX[b] for b in temp]      # SubWord: S-Box
            temp[0] ^= RCON[i // 4 - 1]         # Rcon XOR
        w.append([w[i-4][j] ^ temp[j] for j in range(4)])

    return w


def get_round_key(w, round_num) -> list:
    """Verilen tur numarası için 4×4 tur anahtarı matrisini döndürür."""
    rk = [[0]*4 for _ in range(4)]
    for col in range(4):
        word = w[round_num * 4 + col]
        for row in range(4):
            rk[row][col] = word[row]
    return rk


# Test
_test_key = b'SuperGizliAnahtr'
_w = key_expansion(_test_key)
print(f"Üretilen kelime sayısı : {len(_w)}  (beklenen: 44)")
print(f"Tur 0 anahtarı (ilk kelime) : {_w[0]}")
print(f"Tur 10 anahtarı (son kelime): {_w[43]}")


Üretilen kelime sayısı : 44  (beklenen: 44)
Tur 0 anahtarı (ilk kelime) : [83, 117, 112, 101]
Tur 10 anahtarı (son kelime): [34, 113, 85, 79]


<a id='5'></a>
### Şifreleme Dönüşümleri

Her AES turu dört temel dönüşümden oluşur:

| Dönüşüm | Amaç | Son Tur? |
|---------|------|----------|
| **AddRoundKey** | Tur anahtarıyla XOR — gizlilik | ✅ |
| **SubBytes** | S-Box ile doğrusal olmayan ikame | ✅ |
| **ShiftRows** | Satırları sola kaydır — transpozisyon difüzyonu | ✅ |
| **MixColumns** | GF(2⁸)'de sütun karıştırma — güçlü difüzyon | ❌ |


In [ ]:
def add_round_key(state, round_key):
    """Her hücreyi tur anahtarıyla XOR'lar. XOR kendi tersidir."""
    return [[state[r][c] ^ round_key[r][c] for c in range(4)] for r in range(4)]


def sub_bytes(state):
    """
    Her baytı S-Box tablosuna göre değiştirir.
    → Doğrusal olmayan karışıklık (confusion) sağlar.
    """
    return [[SBOX[state[r][c]] for c in range(4)] for r in range(4)]


def shift_rows(state):
    """
    Satırları sola döndürür (transpozisyon difüzyonu):
      Satır 0 → değişmez
      Satır 1 → 1 bayt sola
      Satır 2 → 2 bayt sola
      Satır 3 → 3 bayt sola
    """
    new_state = [row[:] for row in state]
    for r in range(1, 4):
        new_state[r] = state[r][r:] + state[r][:r]
    return new_state


def mix_columns(state):
    """
    Her sütunu GF(2^8) üzerinde sabit matrisle çarpar.
    Sabit matris:
      [2 3 1 1]
      [1 2 3 1]
      [1 1 2 3]
      [3 1 1 2]
    → Güçlü difüzyon sağlar (1 bayt değişince 4 bayt etkilenir).
    """
    new_state = [[0]*4 for _ in range(4)]
    for c in range(4):
        s = [state[r][c] for r in range(4)]
        new_state[0][c] = gmul(s[0],2) ^ gmul(s[1],3) ^ s[2]          ^ s[3]
        new_state[1][c] = s[0]          ^ gmul(s[1],2) ^ gmul(s[2],3) ^ s[3]
        new_state[2][c] = s[0]          ^ s[1]          ^ gmul(s[2],2) ^ gmul(s[3],3)
        new_state[3][c] = gmul(s[0],3) ^ s[1]          ^ s[2]          ^ gmul(s[3],2)
    return new_state


#### AES-128 Şifreleme Fonksiyonu

**Akış:** Ön Tur (AddRoundKey) → 9 Ana Tur → Son Tur (MixColumns yok)


In [ ]:
def aes_encrypt(plaintext: bytes, key: bytes, verbose=True) -> bytes:
    """
    AES-128 şifreleme.
    verbose=True → her adımı ekrana yazdırır (eğitim modu).
    """
    assert len(plaintext) == 16, "Düz metin 16 bayt olmalı!"
    assert len(key) == 16,       "Anahtar 16 bayt (128-bit) olmalı!"

    w     = key_expansion(key)
    state = bytes_to_state(plaintext)

    if verbose:
        print("\n" + "="*50)
        print("  AES-128 ŞİFRELEME — ADIM ADIM")
        print("="*50)
        print(f"\n  Düz Metin : {plaintext.decode(errors='replace')}")
        print(f"  Anahtar   : {key.decode(errors='replace')}")
        print_step("Başlangıç Durumu", state, "Metin 4x4 matrise yerleştirildi")

    # ── Ön Tur: AddRoundKey ──
    rk    = get_round_key(w, 0)
    state = add_round_key(state, rk)
    if verbose:
        print_step("Tur 0 → AddRoundKey", state, "İlk tur anahtarıyla XOR")

    # ── Ana Turlar 1–9 ──
    for round_num in range(1, 10):
        if verbose:
            print(f"\n  {'─'*48}")
            print(f"  TUR {round_num}")
            print(f"  {'─'*48}")

        state = sub_bytes(state)
        if verbose: print_step(f"Tur {round_num} → SubBytes",    state, "Her bayt S-Box'tan geçirildi")

        state = shift_rows(state)
        if verbose: print_step(f"Tur {round_num} → ShiftRows",   state, "Satırlar sola kaydırıldı")

        state = mix_columns(state)
        if verbose: print_step(f"Tur {round_num} → MixColumns",  state, "Sütunlar GF(2^8)'de karıştırıldı")

        rk    = get_round_key(w, round_num)
        state = add_round_key(state, rk)
        if verbose: print_step(f"Tur {round_num} → AddRoundKey", state, f"{round_num}. tur anahtarıyla XOR")

    # ── Son Tur 10: MixColumns YOK ──
    if verbose:
        print(f"\n  {'─'*48}")
        print(f"  TUR 10 (Son Tur — MixColumns yok)")
        print(f"  {'─'*48}")

    state = sub_bytes(state)
    if verbose: print_step("Tur 10 → SubBytes", state)

    state = shift_rows(state)
    if verbose: print_step("Tur 10 → ShiftRows", state)

    rk    = get_round_key(w, 10)
    state = add_round_key(state, rk)
    if verbose: print_step("Tur 10 → AddRoundKey", state, "Son tur anahtarıyla XOR")

    ciphertext = state_to_bytes(state)

    if verbose:
        print(f"\n  {'='*50}")
        print(f"  ŞİFRELİ METİN (hex): {ciphertext.hex().upper()}")
        print(f"  {'='*50}\n")

    return ciphertext


<a id='6'></a>
###  Şifre Çözme — Ters Dönüşümler

Şifre çözme, şifrelemein tam tersidir. Tur anahtarları ters sırada uygulanır.

| Ters Dönüşüm | Karşılığı |
|---|---|
| `inv_sub_bytes` | InvS-Box |
| `inv_shift_rows` | Sağa döndür |
| `inv_mix_columns` | Ters matris (9,11,13,14 katsayıları) |
| `add_round_key` | XOR kendi tersidir |


In [ ]:
def inv_sub_bytes(state):
    """Her bayta Ters S-Box (INV_SBOX) uygular."""
    return [[INV_SBOX[state[r][c]] for c in range(4)] for r in range(4)]


def inv_shift_rows(state):
    """Satırları sağa döndürür (ShiftRows'un tersi)."""
    new_state = [row[:] for row in state]
    for r in range(1, 4):
        new_state[r] = state[r][4-r:] + state[r][:4-r]
    return new_state


def inv_mix_columns(state):
    """
    Ters MixColumns — ters sabit matris (9, 11, 13, 14 katsayıları):
      [14  11  13   9]
      [ 9  14  11  13]
      [13   9  14  11]
      [11  13   9  14]
    """
    new_state = [[0]*4 for _ in range(4)]
    for c in range(4):
        s = [state[r][c] for r in range(4)]
        new_state[0][c] = gmul(s[0],14) ^ gmul(s[1],11) ^ gmul(s[2],13) ^ gmul(s[3],9)
        new_state[1][c] = gmul(s[0],9)  ^ gmul(s[1],14) ^ gmul(s[2],11) ^ gmul(s[3],13)
        new_state[2][c] = gmul(s[0],13) ^ gmul(s[1],9)  ^ gmul(s[2],14) ^ gmul(s[3],11)
        new_state[3][c] = gmul(s[0],11) ^ gmul(s[1],13) ^ gmul(s[2],9)  ^ gmul(s[3],14)
    return new_state


def aes_decrypt(ciphertext: bytes, key: bytes, verbose=True) -> bytes:
    """
    AES-128 şifre çözme.
    Tur anahtarları ters sırada: 10 → 9 → ... → 0
    """
    assert len(ciphertext) == 16, "Şifreli metin 16 bayt olmalı!"
    assert len(key) == 16,        "Anahtar 16 bayt (128-bit) olmalı!"

    w     = key_expansion(key)
    state = bytes_to_state(ciphertext)

    if verbose:
        print("\n" + "="*50)
        print("  AES-128 ŞİFRE ÇÖZME — ADIM ADIM")
        print("="*50)
        print(f"\n  Şifreli Metin : {ciphertext.hex().upper()}")
        print_step("Başlangıç Durumu", state, "Şifreli metin 4x4 matrise yerleştirildi")

    rk    = get_round_key(w, 10)
    state = add_round_key(state, rk)
    if verbose: print_step("Tur 10 → AddRoundKey", state, "10. tur anahtarıyla XOR")

    for round_num in range(9, 0, -1):
        if verbose:
            print(f"\n  {'─'*48}")
            print(f"  TERS TUR {round_num}")
            print(f"  {'─'*48}")

        state = inv_shift_rows(state)
        if verbose: print_step(f"Tur {round_num} → InvShiftRows",   state, "Satırlar sağa kaydırıldı")

        state = inv_sub_bytes(state)
        if verbose: print_step(f"Tur {round_num} → InvSubBytes",    state, "Her bayt ters S-Box'tan geçirildi")

        rk    = get_round_key(w, round_num)
        state = add_round_key(state, rk)
        if verbose: print_step(f"Tur {round_num} → AddRoundKey",    state, f"{round_num}. tur anahtarıyla XOR")

        state = inv_mix_columns(state)
        if verbose: print_step(f"Tur {round_num} → InvMixColumns",  state, "Ters sütun karıştırma")

    if verbose:
        print(f"\n  {'─'*48}")
        print(f"  SON ADIM")
        print(f"  {'─'*48}")

    state = inv_shift_rows(state)
    if verbose: print_step("InvShiftRows", state)

    state = inv_sub_bytes(state)
    if verbose: print_step("InvSubBytes", state)

    rk    = get_round_key(w, 0)
    state = add_round_key(state, rk)
    if verbose: print_step("AddRoundKey (Tur 0)", state, "İlk tur anahtarıyla XOR")

    plaintext = state_to_bytes(state)

    if verbose:
        print(f"\n  {'='*50}")
        print(f"  ÇÖZÜLEN METİN: {plaintext.decode(errors='replace')}")
        print(f"  {'='*50}\n")

    return plaintext


<a id='7'></a>
### 7️⃣ PKCS#7 Dolgu ve Kullanıcı Arayüzü

AES blok şifreleyici olduğundan metin uzunluğu **16 baytın tam katı** olmalıdır.
**PKCS#7**: eksik `n` bayt varsa `n` değerinde `n` adet bayt eklenir. (Örneğin 3 bayt eksikse `03 03 03` eklenir.)


In [ ]:
def pkcs7_pad(data: bytes) -> bytes:
    """Veriyi PKCS#7 standardına göre 16 baytın katına tamamlar."""
    pad_len = 16 - (len(data) % 16)
    return data + bytes([pad_len] * pad_len)


def pkcs7_unpad(data: bytes) -> bytes:
    """PKCS#7 dolguyu kaldırır."""
    pad_len = data[-1]
    return data[:-pad_len]


def menu():
    print("\n" + "="*50)
    print("  AES-128 ŞİFRELEME ARACI")
    print("="*50)
    print("  1) Şifrele")
    print("  2) Şifre Çöz")
    print("  3) Çıkış")
    print("="*50)
    return input("  Seçim: ").strip()


def get_key(prompt) -> bytes:
    """Anahtarı alır; 16 bayttan kısa ise boşlukla tamamlar, uzunsa kırpar."""
    val = input(prompt).encode()
    if len(val) < 16:
        val = val.ljust(16)
    return val[:16]


def main():
    while True:
        secim = menu()
        if secim == "1":
            print("\n  [Metin herhangi bir uzunlukta olabilir; otomatik dolgu uygulanır]\n")
            metin_str = input("  Düz metin girin  : ").encode()
            anahtar   = get_key("  Anahtar girin (16 karakter): ")
            verbose   = input("  Adım adım göster? (e/h): ").strip().lower() == "e"
            padded    = pkcs7_pad(metin_str)
            bloklar   = [padded[i:i+16] for i in range(0, len(padded), 16)]
            print(f"\n  Toplam {len(bloklar)} blok şifrelenecek.\n")
            tum_sifreli = b""
            for idx, blok in enumerate(bloklar):
                print(f"  ── Blok {idx+1} ──")
                tum_sifreli += aes_encrypt(blok, anahtar, verbose=verbose)
            print(f"\n  Tüm şifreli metin (hex): {tum_sifreli.hex().upper()}")

        elif secim == "2":
            print("\n  [Şifreli metin hex olarak girilmeli]\n")
            hex_metin = input("  Şifreli metin (hex): ").strip().replace(" ", "")
            try:
                sifreli = bytes.fromhex(hex_metin)
            except ValueError:
                print("  Hata: Geçersiz hex değeri!"); continue
            if len(sifreli) % 16 != 0:
                print("  Hata: Şifreli metin 16 baytın katı olmalı!"); continue
            anahtar = get_key("  Anahtar girin (16 karakter): ")
            verbose = input("  Adım adım göster? (e/h): ").strip().lower() == "e"
            bloklar = [sifreli[i:i+16] for i in range(0, len(sifreli), 16)]
            tum_cozulmus = b""
            for idx, blok in enumerate(bloklar):
                print(f"\n  ── Blok {idx+1} ──")
                tum_cozulmus += aes_decrypt(blok, anahtar, verbose=verbose)
            tum_cozulmus = pkcs7_unpad(tum_cozulmus)
            print(f"\n  Çözülen metin: {tum_cozulmus.decode(errors='replace')}")

        elif secim == "3":
            print("\n  Çıkılıyor...\n"); break
        else:
            print("  Geçersiz seçim!")


<a id='8'></a>
###  Demo: AES-128 Şifrele & Çöz

> Aşağıdaki hücreyi çalıştırarak şifreleme ve şifre çözme adımlarını canlı izleyebilirsiniz.
> `verbose=False` yaparsanız yalnızca özet çıktı görürsünüz.


In [ ]:
METIN   = "Merhaba Dünya!!".encode('utf-8')
ANAHTAR = b"SuperGizliAnahtr"

# ── ŞİFRELEME ──
sifreli = aes_encrypt(METIN, ANAHTAR, verbose=True)

# ── ŞİFRE ÇÖZME ──
cozulmus = aes_decrypt(sifreli, ANAHTAR, verbose=True)

# ── ÖZET ──
print("\n" + "="*50)
print("  ÖZET")
print("="*50)
print(f"  Orijinal  : {METIN.decode()}")
print(f"  Şifreli   : {sifreli.hex().upper()}")
print(f"  Çözülmüş  : {cozulmus.decode()}")
print(f"  Doğrulama : {'✓ BAŞARILI' if cozulmus == METIN else '✗ HATA'}")
print("="*50)


  AES-128 ŞİFRELEME — ADIM ADIM

  Düz Metin : Merhaba Dünya!!
  Anahtar   : SuperGizliAnahtr

  ┌─ Başlangıç Durumu ────────────────────────
  │  Metin 4x4 matrise yerleştirildi
  │
  │   4D  61  44  79  
  │   65  62  C3  61  
  │   72  61  BC  21  
  │   68  20  6E  21  
  └─────────────────────────────────────────────

  ┌─ Tur 0 → AddRoundKey ─────────────────────
  │  İlk tur anahtarıyla XOR
  │
  │   1E  13  28  18  
  │   10  25  AA  09  
  │   02  08  FD  55  
  │   0D  5A  00  53  
  └─────────────────────────────────────────────

  ────────────────────────────────────────────────
  TUR 1
  ────────────────────────────────────────────────

  ┌─ Tur 1 → SubBytes ────────────────────────
  │  Her bayt S-Box'tan geçirildi
  │
  │   72  7D  34  AD  
  │   CA  3F  AC  01  
  │   77  30  54  FC  
  │   D7  BE  63  ED  
  └─────────────────────────────────────────────

  ┌─ Tur 1 → ShiftRows ───────────────────────
  │  Satırlar sola kaydırıldı
  │
  │   72  7D  34  AD  
  │   3F  

<a id='app-aes'></a>
---
## 🧩 Uygulama Örneği: Güvenli Mesaj Şifreleme (CBC Modu)

Yukarıdaki `aes_encrypt` fonksiyonu yalnızca **tek bir 128-bit bloğu** şifreler. Ancak gerçek mesaj ve dosyalar genellikle 16 baytın katı değildir ve tek bloktan büyüktür. Bu nedenle AES pratikte bir **çalışma modu** ve **dolgu (padding)** ile birlikte kullanılır.

Burada **CBC (Cipher Block Chaining)** modu uygulanmıştır:
- Her blok, şifrelenmeden önce bir önceki şifreli blok ile **XOR**'lanır (zincirleme).
- İlk blok için rastgele bir **IV (başlatma vektörü)** kullanılır.
- Aynı metin her şifrelendiğinde, IV sayesinde **farklı** bir çıktı üretilir → örüntü sızıntısı engellenir.

> Bu örnek, daha önce tanımlanan `aes_encrypt`, `aes_decrypt`, `pkcs7_pad` ve `pkcs7_unpad` fonksiyonlarını yeniden kullanır.


In [ ]:
import os

BLOK = 16  # AES blok boyutu (bayt)


def _xor_blok(a: bytes, b: bytes) -> bytes:
    """İki bloğu bayt bayt XOR'lar."""
    return bytes(x ^ y for x, y in zip(a, b))


def cbc_sifrele(duz_metin: bytes, anahtar: bytes) -> bytes:
    """
    CBC modu ile şifreleme.
    Çıktı: IV (16 bayt) + şifreli bloklar.
    """
    iv     = os.urandom(BLOK)             # her şifrelemede benzersiz IV
    veri   = pkcs7_pad(duz_metin)         # 16'nın katına tamamla
    onceki = iv
    cikti  = iv
    for i in range(0, len(veri), BLOK):
        blok    = veri[i:i + BLOK]
        karisik = _xor_blok(blok, onceki)            # zincirleme
        sifreli = aes_encrypt(karisik, anahtar, verbose=False)
        cikti  += sifreli
        onceki  = sifreli
    return cikti


def cbc_coz(sifreli_veri: bytes, anahtar: bytes) -> bytes:
    """CBC çözme: ilk 16 bayt IV'dir."""
    iv     = sifreli_veri[:BLOK]
    veri   = sifreli_veri[BLOK:]
    onceki = iv
    cikti  = b""
    for i in range(0, len(veri), BLOK):
        blok    = veri[i:i + BLOK]
        cozulen = aes_decrypt(blok, anahtar, verbose=False)
        cikti  += _xor_blok(cozulen, onceki)         # orijinal blok geri gelir
        onceki  = blok
    return pkcs7_unpad(cikti)


# ── SENARYO ──────────────────────────────────────────────
anahtar = b"SuperGizliAnahtr"                       # 16 bayt (128-bit)
mesaj   = "Gizli mesaj: hesap bakiyesi 12500 TL. Bu bilgi gizlidir!".encode("utf-8")

print("=" * 55)
print("  AES-128 / CBC — GÜVENLİ MESAJ ŞİFRELEME")
print("=" * 55)
print(f"  Orijinal mesaj : {mesaj.decode()}")
print(f"  Uzunluk        : {len(mesaj)} bayt (16'nın katı değil → dolgu gerekir)")

# Şifreleme
sifreli = cbc_sifrele(mesaj, anahtar)
print(f"\n  IV             : {sifreli[:BLOK].hex()}")
print(f"  Şifreli metin  : {sifreli[BLOK:].hex()}")
print(f"  Toplam uzunluk : {len(sifreli)} bayt")

# Çözme (doğru anahtar)
cozulen = cbc_coz(sifreli, anahtar)
print(f"\n  Çözülen mesaj  : {cozulen.decode()}")
print(f"  Doğrulama      : {'✓ BAŞARILI' if cozulen == mesaj else '✗ HATA'}")

# CBC'nin gücü: aynı metin → farklı çıktı
s1 = cbc_sifrele(mesaj, anahtar)
s2 = cbc_sifrele(mesaj, anahtar)
print(f"\n  Aynı metin iki kez şifrelendi:")
print(f"    1. çıktı (ilk blok): {s1[BLOK:2*BLOK].hex()}")
print(f"    2. çıktı (ilk blok): {s2[BLOK:2*BLOK].hex()}")
print(f"    Çıktılar farklı mı? : {'✓ EVET (IV sayesinde örüntü sızmaz)' if s1 != s2 else '✗ HAYIR'}")
print("=" * 55)


---
## 🔷 BÖLÜM 2: ECDSA P-256 — Eliptik Eğri Dijital İmzası

> **ECDSA**, bir mesajın gerçekten senden geldiğini kanıtlayan asimetrik dijital imza standardıdır.
> Banka havalesi, TLS sertifikaları, Bitcoin işlemleri ve JWT tokenleri bu algoritmayı kullanır.

### Nasıl Çalışır? (3 adım)
1. **Özel anahtar** (gizli): rastgele büyük bir sayı `d`
2. **Açık anahtar** (herkese açık): `Q = d × G` — tersini hesaplamak matematiksel olarak imkânsız
3. **İmzalama**: `(r, s)` çifti üret → **Doğrulama**: açık anahtarla kontrol et

```
Kullanılan kütüphane : yalnızca hashlib (SHA-256)
Geri kalan her şey   : sıfırdan yazıldı
```


<a id='9'></a>
###  P-256 Eğri Parametreleri

P-256 eğrisi `y² = x³ + ax + b (mod p)` denklemini sağlayan noktalar kümesidir.
Tüm parametreler NIST tarafından belirlenmiş ve herkese açıktır — **gizli değildir**.


In [ ]:
import hashlib   # yalnızca SHA-256 için
import os        # kriptografik rastgele sayı üretimi için

# p: Asal modülüs — tüm koordinat hesapları bu sayıya modüler
P = 0xFFFFFFFF00000001000000000000000000000000FFFFFFFFFFFFFFFFFFFFFFFF

# a, b: Eğri katsayıları (y² = x³ + ax + b)
A = 0xFFFFFFFF00000001000000000000000000000000FFFFFFFFFFFFFFFFFFFFFFFC
B = 0x5AC635D8AA3A93E7B3EBBD55769886BC651D06B0CC53B0F63BCE3C3E27D2604B

# G: Üreteç nokta — herkes aynı noktadan başlar
GX = 0x6B17D1F2E12C4247F8BCE6E563A440F277037D812DEB33A0F4A13945D898C296
GY = 0x4FE342E2FE1A7F9B8EE7EB4A7C0F9E162BCE33576B315ECECBB6406837BF51F5
G  = (GX, GY)

# n: Grup mertebesi — anahtar ve imza değerleri [1, n-1] aralığında olmalı
N  = 0xFFFFFFFF00000000FFFFFFFFFFFFFFFFBCE6FAADA7179E84F3B9CAC2FC632551

# INF: Sonsuz nokta (kimlik elemanı — normal sayılardaki 0 gibi)
INF = None

print(f"p bit uzunluğu : {P.bit_length()} bit")
print(f"n bit uzunluğu : {N.bit_length()} bit")
print(f"G.x (ilk 16 hex): {GX:064x}"[:50] + "...")


p bit uzunluğu : 256 bit
n bit uzunluğu : 256 bit
G.x (ilk 16 hex): 6b17d1f2e12c4247f8bce6e563a440f2...


<a id='10'></a>
### Modüler Ters — `mod_inv`

Eliptik eğri aritmetiğinde **bölme işlemi yoktur** — bunun yerine modüler ters kullanılır:

$$a^{-1} \pmod{m} \quad \text{öyle ki} \quad a \cdot a^{-1} \equiv 1 \pmod{m}$$

**Genişletilmiş Öklid Algoritması** ile O(log m) adımda hesaplanır.


In [ ]:
def mod_inv(a, m):
    """
    a sayısının m modülüne göre tersini bulur.
    Genişletilmiş Öklid Algoritması kullanır.

    Kullanım yerleri:
      - Nokta toplamada eğim hesabı  (bölme → ters × çarpma)
      - İmzalamada k'nın tersi
      - Doğrulamada s'nin tersi
    """
    if a == 0:
        raise ZeroDivisionError("Sıfırın modüler tersi olmaz!")

    old_r, r = a % m, m
    old_s, s = 1, 0

    while r != 0:
        q        = old_r // r
        old_r, r = r, old_r - q * r
        old_s, s = s, old_s - q * s

    return old_s % m   # negatif çıkabilir → % m ile pozitife çevir


# Doğrulama: 3 × 5 = 15 = 2×7 + 1 → mod 7 = 1
assert mod_inv(3, 7) == 5, "Hata!"
print(f"mod_inv(3, 7) = {mod_inv(3, 7)}  ✓  (3 × 5 mod 7 = {3*5 % 7})")


mod_inv(3, 7) = 5  ✓  (3 × 5 mod 7 = 1)


<a id='11'></a>
### Eliptik Eğri Nokta Aritmetiği

#### point_add — İki Farklı Nokta Toplama
P1 ve P2'yi birleştiren doğrunun eğriyle 3. kesişim noktası bulunur, x eksenine yansıtılır.

$$\lambda = \frac{y_2 - y_1}{x_2 - x_1} \pmod{p}, \quad x_3 = \lambda^2 - x_1 - x_2, \quad y_3 = \lambda(x_1 - x_3) - y_1$$

#### point_double — Nokta İkileme (2P)
Aynı nokta kendisiyle toplanınca tanjant yöntemi uygulanır:

$$\lambda = \frac{3x_1^2 + a}{2y_1} \pmod{p}$$

#### scalar_mult — Double-and-Add
k × G'yi k kez döngüyle hesaplamak 2²⁵⁶ adım alır. Bunun yerine k'yı binary'de işleyerek **~256 adımda** tamamlanır.


In [ ]:
def point_add(P1, P2):
    """
    İki eğri noktasını toplar: P1 + P2
    Özel durumlar:
      P1 = INF → P2,  P2 = INF → P1
      P1 = P2  → point_double,  P1 = -P2 → INF
    """
    if P1 is INF: return P2
    if P2 is INF: return P1

    x1, y1 = P1
    x2, y2 = P2

    if x1 == x2:
        if y1 == y2: return point_double(P1)  # aynı nokta
        else:        return INF                # ters nokta

    lam = (y2 - y1) * mod_inv(x2 - x1, P) % P
    x3  = (lam * lam - x1 - x2) % P
    y3  = (lam * (x1 - x3) - y1) % P
    return (x3, y3)


def point_double(P1):
    """
    Bir noktayı iki katına alır: 2 × P1
    Tanjant yöntemi — eğrinin türevinden eğim hesaplanır.
    """
    if P1 is INF: return INF

    x1, y1 = P1
    lam = (3 * x1 * x1 + A) * mod_inv(2 * y1, P) % P
    x3  = (lam * lam - 2 * x1) % P
    y3  = (lam * (x1 - x3) - y1) % P
    return (x3, y3)


def scalar_mult(k, nokta):
    """
    Skaler çarpım: k × nokta  (Double-and-Add algoritması)

    k'yı binary'de işle:
      Bit 1 ise → noktayı sonuca ekle (point_add)
      Her adımda → noktayı ikile (point_double)
      → ~256 adımda tamamlanır (2^256 adım yerine!)
    """
    sonuc    = INF    # kimlik elemanı (sıfır)
    toplanan = nokta  # her adımda iki katına alınacak değer

    while k:
        if k & 1:                                # en sağ bit 1 mi?
            sonuc = point_add(sonuc, toplanan)   # ekle
        toplanan = point_double(toplanan)        # ikile
        k >>= 1                                  # bir sonraki bit

    return sonuc


# Hızlı test: 2 × G = G + G
_2G_add    = point_add(G, G)
_2G_double = point_double(G)
assert _2G_add == _2G_double, "point_add ve point_double tutarsız!"
print("point_add(G,G) == point_double(G)  ✓")


point_add(G,G) == point_double(G)  ✓


<a id='12'></a>
### Yardımcı Fonksiyonlar


In [ ]:
def sha256_int(veri: bytes) -> int:
    """
    SHA-256 özeti alır ve tam sayıya çevirir.
    → İmzalamada mesaj özeti (e) olarak kullanılır.
    """
    return int.from_bytes(hashlib.sha256(veri).digest(), "big")


def guvenli_rastgele() -> int:
    """
    [1, n-1] aralığında kriptografik olarak güvenli rastgele sayı üretir.
    os.urandom → işletim sisteminin donanım RNG'sini kullanır.
    Python'un random modülü kriptografi için yeterince güvenli DEĞİLDİR!
    """
    while True:
        k = int.from_bytes(os.urandom(32), "big")
        if 1 <= k <= N - 1:
            return k


def cizgi(baslik=""):
    if baslik: print(f"\n  --- {baslik} ---")
    else:      print("  " + "-" * 56)


# Test
_e = sha256_int(b"test")
print(f"SHA-256('test') = {_e:064x}")
print(f"Bit uzunluğu    : {_e.bit_length()} bit")


SHA-256('test') = 9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08
Bit uzunluğu    : 256 bit


<a id='13'></a>
### Anahtar Üretimi

| Anahtar | Tanım | Gizlilik |
|---------|-------|----------|
| **Özel anahtar (d)** | `[1, n-1]` aralığında rastgele sayı | 🔒 Gizli |
| **Açık anahtar (Q)** | `Q = d × G` | 🌐 Herkese açık |

> **Neden güvenli?** Q bilinse de `d = Q / G` hesabı **Ayrık Logaritma Problemi** nedeniyle evrenin ömründen uzun sürer.


In [ ]:
def anahtar_uret():
    """Bir çift anahtar üretir: (ozel_anahtar, acik_anahtar)"""

    print("\n" + "=" * 60)
    print("  ADIM 1: ANAHTAR ÜRETİMİ")
    print("=" * 60)
    print("""
  Eliptik eğri üzerinde iki anahtar üretiyoruz:

  [*] Özel anahtar (d) : Rastgele büyük bir sayı. GİZLİ kalır.
  [*] Açık anahtar (Q) : d × G işleminin sonucu. Herkese verilebilir.

  Güvenlik neden sağlanıyor?
  Q = d × G biliniyor ama buradan d bulunamıyor.
  256-bit eğride tersini hesaplamak evrenin ömründen uzun sürer.
    """)

    d = guvenli_rastgele()
    print(f"  [1] Özel anahtar (d) üretildi:")
    print(f"      d = {d:064x}")
    print(f"      Bu sayı {d.bit_length()} bit uzunluğunda.")

    print(f"\n  [2] Açık anahtar hesaplanıyor: Q = d × G")
    Q = scalar_mult(d, G)

    print(f"\n      Q.x = {Q[0]:064x}")
    print(f"      Q.y = {Q[1]:064x}")
    print("""
  +----------------------------------------------------------+
  |  Özel anahtar (d) → GİZLİ TUT, kimseyle paylaşma      |
  |  Açık anahtar  (Q) → Herkese verebilirsin              |
  +----------------------------------------------------------+
    """)
    return d, Q


<a id='14'></a>
### İmzalama

**İmzalama adımları:**

| Adım | İşlem |
|------|-------|
| 1 | `e = SHA-256(mesaj)` — mesaj özetini al |
| 2 | `k` = rastgele tek kullanımlık gizli sayı (**her imzada farklı!**) |
| 3 | `R = k × G`,  `r = R.x mod n` |
| 4 | `s = k⁻¹ × (e + r×d) mod n` |
| 5 | İmza = `(r, s)` |

> ⚠️ **Kritik:** Aynı k iki farklı imzada kullanılırsa özel anahtar hesaplanabilir! Gerçek sistemlerde RFC 6979 kullanılır.


In [ ]:
def imzala(mesaj: bytes, ozel_anahtar: int):
    """Mesajı özel anahtarla imzalar. Sonuç: (r, s) çifti"""

    print("\n" + "=" * 60)
    print("  ADIM 2: İMZALAMA")
    print("=" * 60)
    print(f"""
  Mesajı özel anahtarımızla imzalıyoruz.
  Çıktı: (r, s) çifti. Bu iki sayı bizim "imzamız"dır.

  Mesaj: "{mesaj.decode(errors='replace')}"
    """)

    # Adım 1: Mesaj özeti
    cizgi("Adım 1: Mesaj özetini al (SHA-256)")
    e = sha256_int(mesaj)
    print(f"  e = SHA-256(mesaj) = {e:064x}")

    while True:
        # Adım 2: Rastgele k
        cizgi("Adım 2: Rastgele k seç (tek kullanımlık)")
        k = guvenli_rastgele()
        print(f"  k = {k:064x}")
        print("  UYARI: Aynı k iki kez kullanılırsa özel anahtar ele geçirilebilir!")

        # Adım 3: R = k × G
        cizgi("Adım 3: R = k × G,  r = R.x mod n")
        R = scalar_mult(k, G)
        r = R[0] % N
        print(f"  R.x = {R[0]:064x}")
        print(f"  r   = {r:064x}")
        if r == 0:
            print("  r = 0 çıktı — yeni k deneniyor..."); continue

        # Adım 4: s = k⁻¹ × (e + r×d)
        cizgi("Adım 4: s = k⁻¹ × (e + r×d) mod n")
        k_inv = mod_inv(k, N)
        s     = k_inv * (e + r * ozel_anahtar) % N
        print(f"  k⁻¹ = {k_inv:064x}")
        print(f"  s   = {s:064x}")
        if s == 0:
            print("  s = 0 çıktı — yeni k deneniyor..."); continue
        break

    print(f"""
  +----------------------------------------------------------+
  |  İMZA (r, s) çifti:                                    |
  |  r = {r:064x}  |
  |  s = {s:064x}  |
  |  Bu iki sayı = dijital imzan.                          |
  +----------------------------------------------------------+
    """)
    return (r, s)


<a id='15'></a>
###  Doğrulama

**Doğrulayan tarafta yalnızca şunlar vardır:** Mesaj, imza `(r, s)` ve açık anahtar Q
→ **Özel anahtara gerek yoktur!**

| Adım | İşlem |
|------|-------|
| 1 | r, s ∈ [1, n-1] mi? |
| 2 | `e = SHA-256(mesaj)` |
| 3 | `w = s⁻¹ mod n` |
| 4 | `u1 = e×w mod n`,  `u2 = r×w mod n` |
| 5 | `X = u1×G + u2×Q` |
| 6 | `X.x mod n == r` → GEÇERLİ |

**Matematiksel ispat:** `k×G = u1×G + u2×Q` çünkü `s = k⁻¹(e + rd)` tanımından `k = s⁻¹(e + rd) = u1 + u2×d`


In [ ]:
def dogrula(mesaj: bytes, imza: tuple, acik_anahtar):
    """
    İmzayı açık anahtarla doğrular.
    Özel anahtara gerek yoktur!
    Sonuç: True (geçerli) veya False (geçersiz)
    """
    r, s = imza

    print("\n" + "=" * 60)
    print("  ADIM 3: DOĞRULAMA")
    print("=" * 60)
    print(f"""
  İmzayı açık anahtarla kontrol ediyoruz.
  Özel anahtara ihtiyaç yok — bu ECDSA'nın gücüdür.

  Elimizde olanlar:
  * Mesaj        : "{mesaj.decode(errors='replace')}"
  * İmza r       : {r:064x}
  * İmza s       : {s:064x}
  * Açık anahtar : (Q.x, Q.y)
    """)

    cizgi("Adım 1: r ve s geçerli aralıkta mı?")
    if not (1 <= r <= N - 1) or not (1 <= s <= N - 1):
        print("  r veya s, [1, n-1] aralığı dışında! → GEÇERSİZ")
        return False
    print("  r ve s aralık kontrolü geçti. ✓")

    cizgi("Adım 2: Mesajın SHA-256 özetini hesapla")
    e = sha256_int(mesaj)
    print(f"  e = {e:064x}")

    cizgi("Adım 3: w = s⁻¹ mod n")
    w = mod_inv(s, N)
    print(f"  w = {w:064x}")

    cizgi("Adım 4: u1 ve u2 hesapla")
    u1 = e * w % N
    u2 = r * w % N
    print(f"  u1 = {u1:064x}")
    print(f"  u2 = {u2:064x}")

    cizgi("Adım 5: X = u1×G + u2×Q hesapla")
    X1 = scalar_mult(u1, G)
    X2 = scalar_mult(u2, acik_anahtar)
    X  = point_add(X1, X2)
    if X is INF:
        print("  X = Sonsuz nokta çıktı → GEÇERSİZ")
        return False
    print(f"  X.x = {X[0]:064x}")

    cizgi("Adım 6: X.x mod n == r ?")
    x_mod_n = X[0] % N
    gecerli = (x_mod_n == r)
    print(f"  X.x mod n = {x_mod_n:064x}")
    print(f"  r         = {r:064x}")
    print(f"  Eşit mi?  → {'EVET ✓' if gecerli else 'HAYIR ✗'}")

    if gecerli:
        print("""
  +----------------------------------------------------------+
  |  DOĞRULAMA SONUCU: İMZA GEÇERLİ ✓                     |
  |  Mesaj değişmemiş. İmza gerçekten bu kişiden.          |
  +----------------------------------------------------------+
        """)
    else:
        print("""
  +----------------------------------------------------------+
  |  DOĞRULAMA SONUCU: İMZA GEÇERSİZ ✗                    |
  |  Mesaj değişmiş ya da imza sahte!                      |
  +----------------------------------------------------------+
        """)
    return gecerli


<a id='16'></a>
### Tahrif Testi

ECDSA'nın en güçlü özelliği: **mesaj tek bir bit değişse imza anında geçersiz olur.**
SHA-256'nın çığ etkisi (avalanche effect) bunu garanti eder.


In [ ]:
def tahrif_testi(mesaj, imza, acik_anahtar):
    """Mesaj değiştirilirse imzanın geçersiz olduğunu kanıtlar."""

    print("\n" + "=" * 60)
    print("  ADIM 4: TAHRİF TESTİ")
    print("=" * 60)
    print(f"""
  Mesajı biraz değiştirip aynı imzayla doğrulamaya çalışıyoruz.
  Beklenen sonuç: doğrulama BAŞARISIZ olmalı.

  Orijinal mesaj  : "{mesaj.decode()}"
    """)

    sahte = mesaj + " [MESAJ TAHRİP EDİLDİ]".encode('utf-8')
    print(f"  Tahrip edilmiş : \"{sahte.decode()}\"")
    print(f"\n  Aynı (r,s) imzasıyla doğrulama yapılıyor...\n")

    sonuc = dogrula(sahte, imza, acik_anahtar)

    print(f"""
  Tahrip edilmiş mesajla sonuç: {"GEÇERLİ (beklenmedik!)" if sonuc else "GEÇERSİZ ✓ (beklenen sonuç)"}

  Neden geçersiz?
  Mesaj değişince SHA-256 özeti (e) tamamen farklı çıktı.
  Bu, u1 ve u2'yi değiştirdi → X noktası değişti.
  X.x artık r'ye eşit değil → imza geçersiz!

  Sonuç: Mesaj 1 karakter bile değişse imza geçersiz olur.
    """)

<a id='17'></a>
###  Demo: Uçtan Uca ECDSA P-256

> Aşağıdaki hücreyi çalıştırarak 4 adımın tamamını canlı izleyebilirsiniz.
> Skaler çarpım hesaplamaları nedeniyle birkaç saniye sürebilir.


In [ ]:
MESAJ = b"Merhaba, bu mesaj ECDSA ile imzalanacak!"

print("\n" + "*" * 60)
print("  ECDSA P-256 — ADIM ADIM AÇIKLAMALI DEMO")
print("*" * 60)
print(f'  İmzalanacak mesaj: "{MESAJ.decode()}"')

# 1) Anahtar üretimi
ozel_anahtar, acik_anahtar = anahtar_uret()

# 2) İmzalama
imza = imzala(MESAJ, ozel_anahtar)

# 3) Doğrulama
sonuc = dogrula(MESAJ, imza, acik_anahtar)

# 4) Tahrif testi
tahrif_testi(MESAJ, imza, acik_anahtar)

# ── Genel Özet ──
print("\n" + "=" * 60)
print("  GENEL ÖZET")
print("=" * 60)
print(f"  Mesaj        : {MESAJ.decode()}")
print(f"  Özel anahtar : {ozel_anahtar:064x}")
print(f"  Açık anahtar :")
print(f"    Q.x = {acik_anahtar[0]:064x}")
print(f"    Q.y = {acik_anahtar[1]:064x}")
print(f"  İmza r       : {imza[0]:064x}")
print(f"  İmza s       : {imza[1]:064x}")
print(f"  Doğrulama    : {'✓ BAŞARILI' if sonuc else '✗ HATA'}")
print("=" * 60)



************************************************************
  ECDSA P-256 — ADIM ADIM AÇIKLAMALI DEMO
************************************************************
  İmzalanacak mesaj: "Merhaba, bu mesaj ECDSA ile imzalanacak!"

  ADIM 1: ANAHTAR ÜRETİMİ

  Eliptik eğri üzerinde iki anahtar üretiyoruz:

  [*] Özel anahtar (d) : Rastgele büyük bir sayı. GİZLİ kalır.
  [*] Açık anahtar (Q) : d × G işleminin sonucu. Herkese verilebilir.

  Güvenlik neden sağlanıyor?
  Q = d × G biliniyor ama buradan d bulunamıyor.
  256-bit eğride tersini hesaplamak evrenin ömründen uzun sürer.
    
  [1] Özel anahtar (d) üretildi:
      d = 359f65232df931007cc45dc5425130a67a904f76978146a2a7c01e3866dd67ff
      Bu sayı 254 bit uzunluğunda.

  [2] Açık anahtar hesaplanıyor: Q = d × G

      Q.x = 8795ce80287b513d45c256b073e52ffe8f05c0a0ff0f14ab9956297ee384f823
      Q.y = f15bd01a7d4c1ca49cc39938ed4e82de9d619afc954a85f2a51813044b93bd93

  +----------------------------------------------------------+
  |  Ö

<a id='app-ecdsa'></a>
---
## 🧩 Uygulama Örneği: Blok Zinciri İşlem İmzalama

ECDSA'nın en yaygın gerçek dünya kullanımlarından biri **blok zinciri işlem imzalamasıdır**. Bitcoin ve Ethereum gibi sistemlerde her işlem, parayı gönderen kullanıcının **özel anahtarıyla** imzalanır; ağdaki düğümler bu imzayı yalnızca **açık anahtarla** doğrular.

Bu sayede:
- Bir kullanıcı yalnızca **kendi** varlıklarını harcayabilir (kimlik doğrulama).
- Yayınlanan bir işlemin içeriği sonradan **değiştirilemez** (bütünlük).

> Bu örnek, daha önce tanımlanan `anahtar_uret`, `imzala` ve `dogrula` fonksiyonlarını yeniden kullanır.


In [ ]:
import hashlib
import json as _json


def adres_uret(acik_anahtar) -> str:
    """Açık anahtardan bir cüzdan adresi türetir (Bitcoin/Ethereum tarzı)."""
    Qx, Qy = acik_anahtar
    ham  = Qx.to_bytes(32, "big") + Qy.to_bytes(32, "big")
    ozet = hashlib.sha256(ham).hexdigest()
    return "0x" + ozet[-40:]              # son 20 bayt → 40 hex karakter


class Cuzdan:
    """Bir kullanıcının özel/açık anahtar çifti ve adresi."""
    def __init__(self, sahip):
        self.sahip = sahip
        self.ozel_anahtar, self.acik_anahtar = anahtar_uret()
        self.adres = adres_uret(self.acik_anahtar)

    def __repr__(self):
        return f"{self.sahip} ({self.adres})"


def islem_govdesi(gonderen, alici, miktar, nonce) -> bytes:
    """İmzalanacak işlem verisini deterministik (sıralı) JSON olarak üretir."""
    islem = {"gonderen": gonderen, "alici": alici, "miktar": miktar, "nonce": nonce}
    return _json.dumps(islem, sort_keys=True, separators=(",", ":")).encode()


def islem_olustur(gonderen_cuzdan, alici_adres, miktar, nonce) -> dict:
    """Gönderen, işlemi kendi ÖZEL anahtarıyla imzalar."""
    govde = islem_govdesi(gonderen_cuzdan.adres, alici_adres, miktar, nonce)
    imza  = imzala(govde, gonderen_cuzdan.ozel_anahtar)
    return {
        "gonderen": gonderen_cuzdan.adres,
        "alici": alici_adres,
        "miktar": miktar,
        "nonce": nonce,
        "acik_anahtar": gonderen_cuzdan.acik_anahtar,
        "imza": imza,
    }


def islem_dogrula(islem: dict) -> bool:
    """Ağdaki bir düğüm, işlemi yalnızca AÇIK anahtarla doğrular."""
    # 1) Açık anahtar gerçekten bu adrese mi ait?
    if adres_uret(islem["acik_anahtar"]) != islem["gonderen"]:
        return False
    # 2) İmza, işlem verisiyle tutarlı mı?
    govde = islem_govdesi(islem["gonderen"], islem["alici"],
                          islem["miktar"], islem["nonce"])
    return dogrula(govde, islem["imza"], islem["acik_anahtar"])


# ── SENARYO ──────────────────────────────────────────────
print("=" * 60)
print("  ECDSA P-256 — BLOK ZİNCİRİ İŞLEM İMZALAMA")
print("=" * 60)

ayse  = Cuzdan("Ayse")
burak = Cuzdan("Burak")
print(f"  Gönderen : {ayse}")
print(f"  Alıcı    : {burak}")

# Meşru işlem
print("\n  → Ayşe, Burak'a 5.0 birim gönderiyor (nonce=1)")
islem = islem_olustur(ayse, burak.adres, 5.0, nonce=1)
print(f"    İmza r : {hex(islem['imza'][0])[:22]}...")
print(f"    Ağ düğümü doğruluyor → {'✓ GEÇERLİ' if islem_dogrula(islem) else '✗ GEÇERSİZ'}")

# Saldırı 1: miktar değiştirme
print("\n  ⚠ Saldırı 1: miktar 5.0 → 5000.0 değiştiriliyor")
sahte = dict(islem); sahte["miktar"] = 5000.0
print(f"    Sonuç → {'✓ GEÇERLİ' if islem_dogrula(sahte) else '✗ GEÇERSİZ (reddedildi)'}")

# Saldırı 2: alıcıyı değiştirme
print("\n  ⚠ Saldırı 2: alıcı, saldırganın adresiyle değiştiriliyor")
saldirgan = Cuzdan("Saldirgan")
sahte2 = dict(islem); sahte2["alici"] = saldirgan.adres
print(f"    Sonuç → {'✓ GEÇERLİ' if islem_dogrula(sahte2) else '✗ GEÇERSİZ (reddedildi)'}")

# Saldırı 3: başkası adına işlem
print("\n  ⚠ Saldırı 3: saldırgan, Ayşe adına işlem üretmeye çalışıyor")
sahte3 = islem_olustur(saldirgan, burak.adres, 5.0, nonce=1)
sahte3["gonderen"] = ayse.adres
print(f"    Sonuç → {'✓ GEÇERLİ' if islem_dogrula(sahte3) else '✗ GEÇERSİZ (reddedildi)'}")
print("=" * 60)
